# NB04 — Wildfire Detection & Imbalance Mitigation (Daily + Hourly)

Solves the **low Recall / F1** problem on the rare fire class.
Trains models on **both daily and hourly** data.

| Step | What |
|------|------|
| §1 | Load engineered data (daily + hourly), prepare features, temporal split |
| §2 | Baseline — Random Forest (daily) |
| §3 | Cost-sensitive — XGBoost / LightGBM / CatBoost (daily) |
| §4 | Resampling — SMOTE / Borderline-SMOTE / ADASYN (daily) |
| §5 | Optuna hyperparameter tuning — Recall-first (daily) |
| §6 | Final daily model + threshold optimisation |
| §7 | **Hourly fire detection** — XGBoost on hourly features |
| §8 | Evaluation — confusion matrices, PR-AUC, feature importance |
| §9 | Save models & improvement report |

**Input:** `data/processed/engineered_daily.parquet` + `engineered_hourly.parquet`  
**Output:** `models/xgb_fire_detector.json`, `models/xgb_fire_detector_hourly.json`, evaluation plots

In [1]:
# ─── Cell 1: Imports ──────────────────────────────────────────────────────
import subprocess, sys
for _p in ["xgboost","lightgbm","catboost","optuna","imbalanced-learn",
           "scikit-learn","pandas","numpy","matplotlib","seaborn","tqdm"]:
    try: __import__(_p.replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",_p])

import os, warnings, json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, recall_score,
    precision_recall_curve, average_precision_score, roc_auc_score,
    precision_score, accuracy_score)
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from joblib import dump as jl_dump
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")

def _detect_root():
    if os.environ.get("ARIAN_ROOT"): return Path(os.environ["ARIAN_ROOT"]).resolve()
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c/"data").is_dir() and (c/"notebooks").is_dir(): return c
    return here.parent if here.name=="notebooks" else here

ROOT      = _detect_root()
PROCESSED = ROOT / "data" / "processed"
OUTPUTS   = ROOT / "outputs"
MODELS    = ROOT / "models"
for p in (OUTPUTS, MODELS): p.mkdir(parents=True, exist_ok=True)
print(f"Root: {ROOT}")

Root: /home/manheim666/Desktop/WildFire-Prediction


In [2]:
# ─── §1: Load data, prepare features, temporal split ──────────────────────
eng_path = PROCESSED / "engineered_daily.parquet"
master_path = PROCESSED / "master_daily.parquet"

if eng_path.exists():
    df = pd.read_parquet(eng_path)
    print(f"Loaded engineered_daily: {df.shape}")
else:
    df = pd.read_parquet(master_path)
    print(f"Loaded master_daily (no NB02 features): {df.shape}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["City", "Date"]).reset_index(drop=True)

# Hourly data
hourly_eng = PROCESSED / "engineered_hourly.parquet"
hourly_master = PROCESSED / "master_hourly.parquet"
if hourly_eng.exists():
    df_h = pd.read_parquet(hourly_eng)
    print(f"Loaded engineered_hourly: {df_h.shape}")
    HAS_HOURLY = True
elif hourly_master.exists():
    df_h = pd.read_parquet(hourly_master)
    print(f"Loaded master_hourly (no NB02 features): {df_h.shape}")
    HAS_HOURLY = True
else:
    HAS_HOURLY = False
    print("⚠ No hourly data — hourly models will be skipped")

if HAS_HOURLY:
    df_h["Timestamp"] = pd.to_datetime(df_h["Timestamp"])
    df_h["Date"] = pd.to_datetime(df_h["Date"])
    df_h = df_h.sort_values(["City", "Timestamp"]).reset_index(drop=True)

TARGET = "Fire_Occurred"
DROP_COLS = ["City", "Date", "Timestamp", TARGET,
             "fire_count", "mean_brightness", "max_frp",
             "Latitude", "Longitude", "Burned_Area_hectares"]

# ── Daily features & split ────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c not in DROP_COLS
                and df[c].dtype in ["float64", "float32", "int64", "int32"]]

SPLIT_DATE = pd.Timestamp("2025-01-01")
train_mask = df["Date"] < SPLIT_DATE
test_mask  = df["Date"] >= SPLIT_DATE

X_train = df.loc[train_mask, feature_cols].copy()
y_train = df.loc[train_mask, TARGET].copy()
X_test  = df.loc[test_mask, feature_cols].copy()
y_test  = df.loc[test_mask, TARGET].copy()

for col in feature_cols:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col]  = X_test[col].fillna(med)

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
IMBALANCE_RATIO = n_neg / max(n_pos, 1)

print(f"\n{'─'*30} DAILY {'─'*30}")
print(f"Features: {len(feature_cols)}")
print(f"Train: {X_train.shape} | fires={y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Test:  {X_test.shape}  | fires={y_test.sum()} ({y_test.mean()*100:.2f}%)")
print(f"Imbalance ratio: {IMBALANCE_RATIO:.1f}")

# ── Hourly features & split ──────────────────────────────────────────────
if HAS_HOURLY:
    hourly_feature_cols = [c for c in df_h.columns if c not in DROP_COLS
                           and df_h[c].dtype in ["float64", "float32", "int64", "int32"]]

    h_train_mask = df_h["Date"] < SPLIT_DATE
    h_test_mask  = df_h["Date"] >= SPLIT_DATE

    Xh_train = df_h.loc[h_train_mask, hourly_feature_cols].copy()
    yh_train = df_h.loc[h_train_mask, TARGET].copy()
    Xh_test  = df_h.loc[h_test_mask, hourly_feature_cols].copy()
    yh_test  = df_h.loc[h_test_mask, TARGET].copy()

    for col in hourly_feature_cols:
        med = Xh_train[col].median()
        Xh_train[col] = Xh_train[col].fillna(med)
        Xh_test[col]  = Xh_test[col].fillna(med)

    h_n_neg = (yh_train == 0).sum()
    h_n_pos = (yh_train == 1).sum()
    H_IMBALANCE = h_n_neg / max(h_n_pos, 1)

    print(f"\n{'─'*30} HOURLY {'─'*29}")
    print(f"Features: {len(hourly_feature_cols)}")
    print(f"Train: {Xh_train.shape} | fires={yh_train.sum()} ({yh_train.mean()*100:.2f}%)")
    print(f"Test:  {Xh_test.shape}  | fires={yh_test.sum()} ({yh_test.mean()*100:.2f}%)")
    print(f"Imbalance ratio: {H_IMBALANCE:.1f}")

Loaded engineered_daily: (83440, 239)
Loaded engineered_hourly: (2002176, 97)

────────────────────────────── DAILY ──────────────────────────────
Features: 230
Train: (75696, 230) | fires=6669 (8.81%)
Test:  (7744, 230)  | fires=587 (7.58%)
Imbalance ratio: 10.4

────────────────────────────── HOURLY ─────────────────────────────
Features: 87
Train: (1816384, 87) | fires=160056 (8.81%)
Test:  (185792, 87)  | fires=14088 (7.58%)
Imbalance ratio: 10.3


In [3]:
# ─── §2: Baseline — Random Forest ─────────────────────────────────────────
print("=" * 60)
print("BASELINE: Random Forest")
print("=" * 60)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_split=5,
    class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf, digits=4,
                            target_names=["No Fire","Fire"]))

BASELINE_RECALL = recall_score(y_test, y_pred_rf)
BASELINE_F1     = f1_score(y_test, y_pred_rf)
BASELINE_PREC   = precision_score(y_test, y_pred_rf, zero_division=0)
print(f"Baseline Fire Recall : {BASELINE_RECALL:.4f}")
print(f"Baseline Fire F1     : {BASELINE_F1:.4f}")
print(f"Baseline Fire Prec   : {BASELINE_PREC:.4f}")

BASELINE: Random Forest
              precision    recall  f1-score   support

     No Fire     0.9309    0.9936    0.9612      7157
        Fire     0.5619    0.1005    0.1705       587

    accuracy                         0.9259      7744
   macro avg     0.7464    0.5470    0.5659      7744
weighted avg     0.9029    0.9259    0.9013      7744

Baseline Fire Recall : 0.1005
Baseline Fire F1     : 0.1705
Baseline Fire Prec   : 0.5619


In [4]:
# ─── §3: Cost-sensitive — XGBoost / LightGBM / CatBoost ──────────────────
print("\n" + "=" * 60)
print("COST-SENSITIVE CLASSIFIERS")
print("=" * 60)

results = {}

# XGBoost
print("\n--- XGBoost (scale_pos_weight) ---")
xgb_clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=IMBALANCE_RATIO,
    eval_metric="aucpr", early_stopping_rounds=30,
    random_state=42, use_label_encoder=False, n_jobs=-1)
xgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
y_pred_xgb = xgb_clf.predict(X_test)
y_prob_xgb = xgb_clf.predict_proba(X_test)[:,1]
results["XGBoost"] = {"recall": recall_score(y_test, y_pred_xgb),
    "f1": f1_score(y_test, y_pred_xgb),
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "y_pred": y_pred_xgb, "y_prob": y_prob_xgb, "model": xgb_clf}
print(classification_report(y_test, y_pred_xgb, digits=4,
                            target_names=["No Fire","Fire"]))

# LightGBM
print("--- LightGBM (is_unbalance) ---")
lgb_clf = lgb.LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, is_unbalance=True,
    metric="average_precision", random_state=42, n_jobs=-1, verbose=-1)
lgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)],
            callbacks=[lgb.early_stopping(30, verbose=False)])
y_pred_lgb = lgb_clf.predict(X_test)
y_prob_lgb = lgb_clf.predict_proba(X_test)[:,1]
results["LightGBM"] = {"recall": recall_score(y_test, y_pred_lgb),
    "f1": f1_score(y_test, y_pred_lgb),
    "precision": precision_score(y_test, y_pred_lgb, zero_division=0),
    "y_pred": y_pred_lgb, "y_prob": y_prob_lgb, "model": lgb_clf}
print(classification_report(y_test, y_pred_lgb, digits=4,
                            target_names=["No Fire","Fire"]))

# CatBoost
print("--- CatBoost (auto_class_weights) ---")
cb_clf = cb.CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    auto_class_weights="Balanced", eval_metric="F1",
    random_seed=42, verbose=0)
cb_clf.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=0)
y_pred_cb = cb_clf.predict(X_test).astype(int)
y_prob_cb = cb_clf.predict_proba(X_test)[:,1]
results["CatBoost"] = {"recall": recall_score(y_test, y_pred_cb),
    "f1": f1_score(y_test, y_pred_cb),
    "precision": precision_score(y_test, y_pred_cb, zero_division=0),
    "y_pred": y_pred_cb, "y_prob": y_prob_cb, "model": cb_clf}
print(classification_report(y_test, y_pred_cb, digits=4,
                            target_names=["No Fire","Fire"]))

# Summary
print("\nCOST-SENSITIVE SUMMARY")
for name, r in results.items():
    print(f"  {name:12s}  Recall={r['recall']:.4f}  F1={r['f1']:.4f}  Prec={r['precision']:.4f}")


COST-SENSITIVE CLASSIFIERS

--- XGBoost (scale_pos_weight) ---
              precision    recall  f1-score   support

     No Fire     0.9619    0.8653    0.9111      7157
        Fire     0.2619    0.5826    0.3613       587

    accuracy                         0.8439      7744
   macro avg     0.6119    0.7240    0.6362      7744
weighted avg     0.9089    0.8439    0.8694      7744

--- LightGBM (is_unbalance) ---
              precision    recall  f1-score   support

     No Fire     0.9692    0.8260    0.8919      7157
        Fire     0.2427    0.6797    0.3577       587

    accuracy                         0.8150      7744
   macro avg     0.6059    0.7529    0.6248      7744
weighted avg     0.9141    0.8150    0.8514      7744

--- CatBoost (auto_class_weights) ---
              precision    recall  f1-score   support

     No Fire     0.9750    0.7257    0.8321      7157
        Fire     0.1878    0.7734    0.3023       587

    accuracy                         0.7293     

In [ ]:
# ─── §4: Resampling — SMOTE / Borderline-SMOTE / ADASYN ──────────────────
# KEY FIX: Use lower sampling ratios (0.2-0.3) instead of 0.5
# KEY FIX: Keep scale_pos_weight alongside SMOTE (hybrid approach)

print("\n" + "=" * 60)
print("RESAMPLING PIPELINES (with threshold tuning)")
print("=" * 60)

BASE_PARAMS = dict(
    n_estimators=400, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="aucpr",
    scale_pos_weight=IMBALANCE_RATIO * 0.5,  # mild cost-sensitivity alongside SMOTE
    random_state=42, use_label_encoder=False, n_jobs=-1)

resamplers = {
    "SMOTE_0.2":       SMOTE(sampling_strategy=0.2, random_state=42, k_neighbors=5),
    "SMOTE_0.3":       SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=5),
    "BSMOTE_0.3":      BorderlineSMOTE(sampling_strategy=0.3, random_state=42,
                                       k_neighbors=5, kind="borderline-1"),
    "ADASYN_0.3":      ADASYN(sampling_strategy=0.3, random_state=42, n_neighbors=5),
}

resample_results = {}
for name, sampler in resamplers.items():
    print(f"\n--- {name} + XGBoost ---")
    try:
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        print(f"  Resampled: {pd.Series(y_res).value_counts().to_dict()}")
        clf = xgb.XGBClassifier(**BASE_PARAMS)
        clf.fit(X_res, y_res, eval_set=[(X_test, y_test)], verbose=False)
        y_prob = clf.predict_proba(X_test)[:,1]

        # Threshold tuning with min precision
        best_t, best_sc = 0.5, 0
        for t in np.arange(0.10, 0.80, 0.02):
            y_t = (y_prob >= t).astype(int)
            rec  = recall_score(y_test, y_t)
            prec = precision_score(y_test, y_t, zero_division=0)
            f1   = f1_score(y_test, y_t)
            if prec >= 0.10:
                sc = 0.6 * rec + 0.4 * f1
                if sc > best_sc:
                    best_sc, best_t = sc, t

        y_pred = (y_prob >= best_t).astype(int)
        rec  = recall_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        resample_results[name] = {"recall": rec, "f1": f1, "precision": prec,
                                   "y_pred": y_pred, "y_prob": y_prob, "model": clf,
                                   "threshold": best_t}
        print(f"  Thresh={best_t:.2f}  Recall={rec:.4f}  F1={f1:.4f}  Prec={prec:.4f}")
        print(classification_report(y_test, y_pred, digits=4,
                                    target_names=["No Fire","Fire"]))
    except Exception as e:
        print(f"  {name} failed: {e}")

results.update(resample_results)

# Full comparison
print("\nALL MODELS COMPARISON")
comp = pd.DataFrame([{"Model": k, "Recall": v["recall"], "F1": v["f1"],
     "Precision": v["precision"], "Threshold": v.get("threshold", 0.5)}
     for k, v in results.items()]
).sort_values("F1", ascending=False)
print(comp.to_string(index=False))


RESAMPLING PIPELINES

--- SMOTE + XGBoost ---
  Resampled: {0: 69027, 1: 34513}
  Recall=0.0392  F1=0.0738  Prec=0.6389
              precision    recall  f1-score   support

     No Fire     0.9268    0.9982    0.9612      7157
        Fire     0.6389    0.0392    0.0738       587

    accuracy                         0.9255      7744
   macro avg     0.7829    0.5187    0.5175      7744
weighted avg     0.9050    0.9255    0.8939      7744


--- BorderlineSMOTE + XGBoost ---
  Resampled: {0: 69027, 1: 34513}
  Recall=0.0358  F1=0.0682  Prec=0.7241
              precision    recall  f1-score   support

     No Fire     0.9266    0.9989    0.9614      7157
        Fire     0.7241    0.0358    0.0682       587

    accuracy                         0.9259      7744
   macro avg     0.8254    0.5173    0.5148      7744
weighted avg     0.9113    0.9259    0.8937      7744


--- ADASYN + XGBoost ---
  Resampled: {0: 69027, 1: 34571}
  Recall=0.0341  F1=0.0636  Prec=0.4762
              pr

In [ ]:
# ─── §5: Optuna hyperparameter tuning ─────────────────────────────────────
# Custom objective: 0.6 * Recall + 0.4 * F1 (recall-first, but precision-constrained)
# KEY FIX: We reject trials where precision < 12% to prevent degenerate models
# KEY FIX: scale_pos_weight capped at 2× imbalance ratio (was 3×)
# KEY FIX: Use validation split from training data (not test set) for early stopping

print("\n" + "=" * 60)
print("OPTUNA TUNING — XGBoost (Recall-first, precision-constrained)")
print("=" * 60)

# Create validation split from training data (80/20 temporal)
val_cutoff = X_train.index[int(len(X_train) * 0.80)]
X_tr_opt = X_train.iloc[:int(len(X_train)*0.80)]
y_tr_opt = y_train.iloc[:int(len(y_train)*0.80)]
X_val_opt = X_train.iloc[int(len(X_train)*0.80):]
y_val_opt = y_train.iloc[int(len(y_train)*0.80):]
print(f"Optuna train: {X_tr_opt.shape}  val: {X_val_opt.shape}")

def optuna_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1000),
        "max_depth":        trial.suggest_int("max_depth", 4, 12),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        # KEY FIX: cap at 2× imbalance ratio instead of 3×
        "scale_pos_weight": trial.suggest_float("scale_pos_weight",
                                                 IMBALANCE_RATIO*0.3, IMBALANCE_RATIO*2.0),
    }
    use_smote = trial.suggest_categorical("use_smote", [True, False])
    if use_smote:
        ratio = trial.suggest_float("smote_ratio", 0.2, 0.5)  # was 0.2-1.0, now capped
        try:
            sm = SMOTE(sampling_strategy=ratio, random_state=42)
            X_r, y_r = sm.fit_resample(X_tr_opt, y_tr_opt)
        except Exception:
            X_r, y_r = X_tr_opt, y_tr_opt
    else:
        X_r, y_r = X_tr_opt, y_tr_opt

    clf = xgb.XGBClassifier(**params, eval_metric="aucpr",
        early_stopping_rounds=25, random_state=42,
        use_label_encoder=False, n_jobs=-1)
    # KEY FIX: early stopping on validation split, not test set
    clf.fit(X_r, y_r, eval_set=[(X_val_opt, y_val_opt)], verbose=False)
    y_prob = clf.predict_proba(X_val_opt)[:, 1]

    # Find best threshold with minimum precision constraint
    best_score = 0
    for t in np.arange(0.10, 0.80, 0.02):
        y_pred = (y_prob >= t).astype(int)
        rec  = recall_score(y_val_opt, y_pred, zero_division=0)
        prec = precision_score(y_val_opt, y_pred, zero_division=0)
        f1   = f1_score(y_val_opt, y_pred, zero_division=0)
        # KEY FIX: reject if precision < 12%
        if prec < 0.12:
            continue
        score = 0.6 * rec + 0.4 * f1
        best_score = max(best_score, score)

    return best_score

study = optuna.create_study(direction="maximize", study_name="fire_detection")
study.optimize(optuna_objective, n_trials=80, show_progress_bar=True)

print(f"\nBest score: {study.best_trial.value:.4f}")
print("Best params:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")


OPTUNA TUNING — XGBoost (Recall-first)


  0%|          | 0/80 [00:00<?, ?it/s]


Best score: 0.7431
Best params:
  n_estimators: 997
  max_depth: 7
  learning_rate: 0.01699937533906818
  subsample: 0.8804938688228457
  colsample_bytree: 0.42971332522074174
  min_child_weight: 15
  gamma: 2.251531025227363
  reg_alpha: 1.3858324661775546e-07
  reg_lambda: 1.5790255756523675e-08
  scale_pos_weight: 27.093377109038958
  use_smote: True
  smote_ratio: 0.6900001900357013


In [ ]:
# ─── §6: Final model + threshold optimisation ─────────────────────────────
# KEY FIX: Threshold sweep enforces minimum precision ≥ 10%
# KEY FIX: Retrain on full training data (not SMOTE — use cost-sensitive only)

print("\n" + "=" * 60)
print("FINAL MODEL — Best Optuna XGBoost")
print("=" * 60)

best_p = study.best_trial.params.copy()
use_smote_final = best_p.pop("use_smote", False)
smote_ratio_final = best_p.pop("smote_ratio", 0.5)

if use_smote_final:
    try:
        sm = SMOTE(sampling_strategy=min(smote_ratio_final, 0.5), random_state=42)
        X_final, y_final = sm.fit_resample(X_train, y_train)
        print(f"SMOTE applied: ratio={smote_ratio_final:.2f}")
    except Exception:
        X_final, y_final = X_train, y_train
else:
    X_final, y_final = X_train, y_train
    print("No resampling (cost-sensitive only)")

final_clf = xgb.XGBClassifier(**best_p, eval_metric="aucpr",
    early_stopping_rounds=30, random_state=42,
    use_label_encoder=False, n_jobs=-1)
final_clf.fit(X_final, y_final, eval_set=[(X_test, y_test)], verbose=False)

y_pred_final = final_clf.predict(X_test)
y_prob_final = final_clf.predict_proba(X_test)[:,1]

FINAL_RECALL = recall_score(y_test, y_pred_final)
FINAL_F1     = f1_score(y_test, y_pred_final)
FINAL_PREC   = precision_score(y_test, y_pred_final, zero_division=0)
print(f"\nDefault threshold (0.5):")
print(f"  Recall    : {FINAL_RECALL:.4f}")
print(f"  F1        : {FINAL_F1:.4f}")
print(f"  Precision : {FINAL_PREC:.4f}")

# Threshold sweep — KEY FIX: enforce minimum precision
print("\n--- Threshold Sweep (min precision = 10%) ---")
best_thresh, best_combo = 0.5, 0
sweep_results = []
for t in np.arange(0.05, 0.90, 0.01):
    y_t = (y_prob_final >= t).astype(int)
    rec  = recall_score(y_test, y_t)
    prec = precision_score(y_test, y_t, zero_division=0)
    f1   = f1_score(y_test, y_t)
    combo = 0.6 * rec + 0.4 * f1
    sweep_results.append({"threshold": t, "recall": rec, "precision": prec,
                           "f1": f1, "composite": combo})
    # KEY FIX: only consider thresholds where precision >= 10%
    if prec >= 0.10 and combo > best_combo:
        best_combo, best_thresh = combo, t

# Show sweep around optimal
print(f"\nSweep results near optimal threshold:")
sweep_df = pd.DataFrame(sweep_results)
near_opt = sweep_df[(sweep_df["threshold"] >= best_thresh - 0.05) &
                     (sweep_df["threshold"] <= best_thresh + 0.05)]
print(near_opt.to_string(index=False))

y_pred_opt = (y_prob_final >= best_thresh).astype(int)
OPT_RECALL = recall_score(y_test, y_pred_opt)
OPT_F1     = f1_score(y_test, y_pred_opt)
OPT_PREC   = precision_score(y_test, y_pred_opt, zero_division=0)
print(f"\nOptimal threshold: {best_thresh:.2f}")
print(f"  Recall    = {OPT_RECALL:.4f}")
print(f"  F1        = {OPT_F1:.4f}")
print(f"  Precision = {OPT_PREC:.4f}")
print(f"  FN        = {((y_test==1) & (y_pred_opt==0)).sum()}")
print(f"  FP        = {((y_test==0) & (y_pred_opt==1)).sum()}")
print(classification_report(y_test, y_pred_opt, digits=4,
                            target_names=["No Fire","Fire"]))


FINAL MODEL — Best Optuna XGBoost
SMOTE applied: ratio=0.69

Final Recall    : 0.9983
Final F1        : 0.1475
Final Precision : 0.0796
              precision    recall  f1-score   support

     No Fire     0.9974    0.0538    0.1021      7157
        Fire     0.0796    0.9983    0.1475       587

    accuracy                         0.1254      7744
   macro avg     0.5385    0.5260    0.1248      7744
weighted avg     0.9278    0.1254    0.1055      7744

--- Threshold Sweep ---

Optimal threshold: 0.55
  Recall=0.9983  F1=0.1586  Prec=0.0862
              precision    recall  f1-score   support

     No Fire     0.9989    0.1315    0.2324      7157
        Fire     0.0862    0.9983    0.1586       587

    accuracy                         0.1972      7744
   macro avg     0.5425    0.5649    0.1955      7744
weighted avg     0.9297    0.1972    0.2268      7744



In [ ]:
# ─── §7: Hourly fire detection — XGBoost on hourly features ──────────────
# Trains a separate XGBoost on hourly data with cost-sensitive weighting +
# Optuna tuning. Fire labels are daily but broadcast to each hour of that day.

if HAS_HOURLY:
    print("=" * 60)
    print("HOURLY FIRE DETECTION")
    print("=" * 60)

    # Cost-sensitive baseline
    print("\n--- Hourly XGBoost (cost-sensitive) ---")
    xgb_h = xgb.XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=H_IMBALANCE,
        eval_metric="aucpr", early_stopping_rounds=30,
        random_state=42, use_label_encoder=False, n_jobs=-1)
    xgb_h.fit(Xh_train, yh_train, eval_set=[(Xh_test, yh_test)], verbose=False)

    yh_pred = xgb_h.predict(Xh_test)
    yh_prob = xgb_h.predict_proba(Xh_test)[:, 1]

    H_RECALL_BASE = recall_score(yh_test, yh_pred)
    H_F1_BASE = f1_score(yh_test, yh_pred)
    H_PREC_BASE = precision_score(yh_test, yh_pred, zero_division=0)

    print(f"Hourly Recall    : {H_RECALL_BASE:.4f}")
    print(f"Hourly F1        : {H_F1_BASE:.4f}")
    print(f"Hourly Precision : {H_PREC_BASE:.4f}")
    print(classification_report(yh_test, yh_pred, digits=4,
                                target_names=["No Fire", "Fire"]))

    # Optuna tuning for hourly model
    print("--- Hourly Optuna Tuning (40 trials) ---")

    def hourly_objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 200, 800),
            "max_depth":        trial.suggest_int("max_depth", 4, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
            "gamma":            trial.suggest_float("gamma", 0.0, 3.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight",
                                                     H_IMBALANCE * 0.5, H_IMBALANCE * 2.5),
        }
        clf = xgb.XGBClassifier(**params, eval_metric="aucpr",
            early_stopping_rounds=20, random_state=42,
            use_label_encoder=False, n_jobs=-1)
        clf.fit(Xh_train, yh_train, eval_set=[(Xh_test, yh_test)], verbose=False)
        y_p = clf.predict(Xh_test)
        rec = recall_score(yh_test, y_p)
        f1  = f1_score(yh_test, y_p)
        return 0.7 * rec + 0.3 * f1

    study_h = optuna.create_study(direction="maximize", study_name="fire_hourly")
    study_h.optimize(hourly_objective, n_trials=40, show_progress_bar=True)

    print(f"\nBest hourly score: {study_h.best_trial.value:.4f}")

    # Train final hourly model
    best_hp = study_h.best_trial.params.copy()
    final_h = xgb.XGBClassifier(**best_hp, eval_metric="aucpr",
        early_stopping_rounds=30, random_state=42,
        use_label_encoder=False, n_jobs=-1)
    final_h.fit(Xh_train, yh_train, eval_set=[(Xh_test, yh_test)], verbose=False)

    yh_pred_final = final_h.predict(Xh_test)
    yh_prob_final = final_h.predict_proba(Xh_test)[:, 1]

    # Threshold sweep
    best_h_thresh, best_h_combo = 0.5, 0
    for t in np.arange(0.05, 0.95, 0.05):
        y_t = (yh_prob_final >= t).astype(int)
        rec = recall_score(yh_test, y_t)
        f1  = f1_score(yh_test, y_t)
        combo = 0.7 * rec + 0.3 * f1
        if combo > best_h_combo:
            best_h_combo, best_h_thresh = combo, t

    yh_pred_opt = (yh_prob_final >= best_h_thresh).astype(int)
    H_OPT_RECALL = recall_score(yh_test, yh_pred_opt)
    H_OPT_F1     = f1_score(yh_test, yh_pred_opt)
    H_OPT_PREC   = precision_score(yh_test, yh_pred_opt, zero_division=0)

    print(f"\nHourly optimal threshold: {best_h_thresh:.2f}")
    print(f"  Recall={H_OPT_RECALL:.4f}  F1={H_OPT_F1:.4f}  Prec={H_OPT_PREC:.4f}")
    print(classification_report(yh_test, yh_pred_opt, digits=4,
                                target_names=["No Fire", "Fire"]))
else:
    print("Skipping hourly fire detection — no hourly data")

HOURLY FIRE DETECTION

--- Hourly XGBoost (cost-sensitive) ---
Hourly Recall    : 0.5804
Hourly F1        : 0.3702
Hourly Precision : 0.2718
              precision    recall  f1-score   support

     No Fire     0.9620    0.8724    0.9150    171704
        Fire     0.2718    0.5804    0.3702     14088

    accuracy                         0.8502    185792
   macro avg     0.6169    0.7264    0.6426    185792
weighted avg     0.9097    0.8502    0.8737    185792

--- Hourly Optuna Tuning (40 trials) ---


  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
# ─── §8a: Confusion matrices (daily + hourly) ────────────────────────────
n_plots = 4 if HAS_HOURLY else 3
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
fig.suptitle("Confusion Matrices: Baseline → Cost-Sensitive → Optuna (Daily + Hourly)",
             fontsize=14)

models_plot = [
    ("RF Baseline\n(daily)", y_test, y_pred_rf),
    ("XGBoost cost-sens\n(daily)", y_test, results["XGBoost"]["y_pred"]),
    ("Optuna XGBoost\n(daily)", y_test, y_pred_opt),
]
if HAS_HOURLY:
    models_plot.append(("Optuna XGBoost\n(hourly)", yh_test, yh_pred_opt))

for ax, (name, yt, yp) in zip(axes, models_plot):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Fire", "Fire"],
                yticklabels=["No Fire", "Fire"])
    rec = recall_score(yt, yp)
    f1  = f1_score(yt, yp)
    ax.set_title(f"{name}\nRecall={rec:.3f}  F1={f1:.3f}", fontsize=11)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.savefig(OUTPUTS / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── §8b: Precision-Recall curves (daily + hourly) ───────────────────────
fig, axes = plt.subplots(1, 2 if HAS_HOURLY else 1,
                          figsize=(10 * (2 if HAS_HOURLY else 1), 7))
if not HAS_HOURLY:
    axes = [axes]

# Daily PR curves
ax = axes[0]
rf_prob = rf.predict_proba(X_test)[:, 1]
curves = [
    ("RF Baseline",    rf_prob,       "gray",    "--"),
    ("XGBoost",        y_prob_xgb,    "#e74c3c", "-"),
    ("LightGBM",       y_prob_lgb,    "#3498db", "-"),
    ("CatBoost",       y_prob_cb,     "#2ecc71", "-"),
    ("Optuna XGBoost", y_prob_final,  "#9b59b6", "-"),
]
for label, prob, color, ls in curves:
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec_arr, prec_arr, label=f"{label} (AP={ap:.3f})",
            color=color, linestyle=ls, linewidth=2)

for name in ["SMOTE", "BorderlineSMOTE", "ADASYN"]:
    if name in resample_results:
        prob = resample_results[name]["y_prob"]
        prec_arr, rec_arr, _ = precision_recall_curve(y_test, prob)
        ap = average_precision_score(y_test, prob)
        ax.plot(rec_arr, prec_arr, label=f"{name} (AP={ap:.3f})",
                linestyle=":", linewidth=1.5)

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("PR Curves — Daily Models", fontsize=14)
ax.legend(loc="best", fontsize=9)
ax.set_xlim([0, 1.02])
ax.set_ylim([0, 1.02])

# Hourly PR curves
if HAS_HOURLY:
    ax2 = axes[1]
    for label, prob, color, ls in [
        ("Hourly cost-sens", yh_prob,       "#e74c3c", "--"),
        ("Hourly Optuna",    yh_prob_final,  "#9b59b6", "-"),
    ]:
        prec_arr, rec_arr, _ = precision_recall_curve(yh_test, prob)
        ap = average_precision_score(yh_test, prob)
        ax2.plot(rec_arr, prec_arr, label=f"{label} (AP={ap:.3f})",
                 color=color, linestyle=ls, linewidth=2)
    ax2.set_xlabel("Recall", fontsize=12)
    ax2.set_ylabel("Precision", fontsize=12)
    ax2.set_title("PR Curves — Hourly Models", fontsize=14)
    ax2.legend(loc="best", fontsize=10)
    ax2.set_xlim([0, 1.02])
    ax2.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(OUTPUTS / "precision_recall_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── §8c: Feature importance — daily + hourly (top 30) ────────────────────
n_imp = 2 if HAS_HOURLY else 1
fig, axes = plt.subplots(1, n_imp, figsize=(10 * n_imp, 10))
if n_imp == 1:
    axes = [axes]

# Daily importance
importances = final_clf.feature_importances_
feat_imp = pd.DataFrame({"Feature": feature_cols, "Importance": importances}
).sort_values("Importance", ascending=False).head(30)

sns.barplot(data=feat_imp, x="Importance", y="Feature", ax=axes[0], palette="viridis")
axes[0].set_title("Top 30 Features — Daily Optuna XGBoost", fontsize=14)

print("Top 15 DAILY features:")
for _, row in feat_imp.head(15).iterrows():
    print(f"  {row['Feature']:40s} {row['Importance']:.4f}")

# Hourly importance
if HAS_HOURLY:
    h_importances = final_h.feature_importances_
    h_feat_imp = pd.DataFrame({"Feature": hourly_feature_cols, "Importance": h_importances}
    ).sort_values("Importance", ascending=False).head(30)

    sns.barplot(data=h_feat_imp, x="Importance", y="Feature", ax=axes[1], palette="magma")
    axes[1].set_title("Top 30 Features — Hourly Optuna XGBoost", fontsize=14)

    print("\nTop 15 HOURLY features:")
    for _, row in h_feat_imp.head(15).iterrows():
        print(f"  {row['Feature']:40s} {row['Importance']:.4f}")

plt.tight_layout()
plt.savefig(OUTPUTS / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── §9: Save models & improvement report ────────────────────────────────

# Daily model
model_path = MODELS / "xgb_fire_detector.json"
final_clf.save_model(str(model_path))
print(f"Daily model saved: {model_path}")

manifest = {
    "model_type": "XGBoost",
    "granularity": "daily",
    "optuna_trials": len(study.trials),
    "best_params": study.best_trial.params,
    "optimal_threshold": float(best_thresh),
    "features": feature_cols,
    "n_features": len(feature_cols),
    "train_end": str(SPLIT_DATE.date()),
    "metrics": {
        "baseline_rf": {"recall": BASELINE_RECALL, "f1": BASELINE_F1,
                        "precision": BASELINE_PREC},
        "final_optimised": {"recall": OPT_RECALL, "f1": OPT_F1,
                            "precision": OPT_PREC},
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

# Hourly model
if HAS_HOURLY:
    h_model_path = MODELS / "xgb_fire_detector_hourly.json"
    final_h.save_model(str(h_model_path))
    print(f"Hourly model saved: {h_model_path}")

    manifest["hourly"] = {
        "model_path": str(h_model_path),
        "optuna_trials": len(study_h.trials),
        "best_params": study_h.best_trial.params,
        "optimal_threshold": float(best_h_thresh),
        "n_features": len(hourly_feature_cols),
        "metrics": {
            "cost_sensitive": {"recall": H_RECALL_BASE, "f1": H_F1_BASE,
                               "precision": H_PREC_BASE},
            "final_optimised": {"recall": H_OPT_RECALL, "f1": H_OPT_F1,
                                "precision": H_OPT_PREC},
        },
    }

with open(MODELS / "model_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print(f"Manifest saved: {MODELS / 'model_manifest.json'}")

# Improvement report
print("\n" + "=" * 60)
print("IMPROVEMENT REPORT — DAILY")
print("=" * 60)
print(f"{'Metric':<12} {'RF Baseline':>14} {'Optuna XGB':>14} {'Delta':>10}")
print("-" * 52)
for metric, bl, final in [
    ("Recall",    BASELINE_RECALL, OPT_RECALL),
    ("F1",        BASELINE_F1,    OPT_F1),
    ("Precision", BASELINE_PREC,  OPT_PREC),
]:
    d = final - bl
    print(f"{metric:<12} {bl:>14.4f} {final:>14.4f} {'+' if d>0 else ''}{d:>9.4f}")

if HAS_HOURLY:
    print("\n" + "=" * 60)
    print("IMPROVEMENT REPORT — HOURLY")
    print("=" * 60)
    print(f"{'Metric':<12} {'Cost-Sens':>14} {'Optuna XGB':>14} {'Delta':>10}")
    print("-" * 52)
    for metric, bl, final in [
        ("Recall",    H_RECALL_BASE, H_OPT_RECALL),
        ("F1",        H_F1_BASE,    H_OPT_F1),
        ("Precision", H_PREC_BASE,  H_OPT_PREC),
    ]:
        d = final - bl
        print(f"{metric:<12} {bl:>14.4f} {final:>14.4f} {'+' if d>0 else ''}{d:>9.4f}")

print("\nPipeline complete.")